# Notebook zur Extraktion der Daten aus dem VPIS

Code zur Extraktion der Veranstaltungsinformationen zu einem Studiengang

In [ ]:
import requests
import xml.etree.ElementTree as ET

mapping = {
    'P': 'Praktikum',
    'V': 'Vorlesung',
    'Ü': 'Übung',
    'Prüfung': 'Prüfung',
    'S': 'Seminar',
    'Tutorium': 'Tutorium',
    'U/Ü': 'Übung',
    'V/Ü': 'Vorlesung/Übung',
    'V/S': 'Vorlesung/Seminar',
    'Ü/S': 'Übung/Seminar',
    'S-online': 'Seminar online'
}

studiengaenge_module = {}

url = "https://vpis.fh-swf.de/WS2025/studiengang.php3?Template=XML&Studiengang=Is-IN-INF&ViewType=Modulverzeichnis"

response = requests.get(url)

if response.status_code == 200:

    inf = {}

    # XML-Inhalt auslesen
    xml_content = response.content

    # XML parsen
    root = ET.fromstring(xml_content)
    modules = root.find('modules')

    # Module durchlaufen
    for module in modules.findall('module'):
        modul_name = module.get('name')
        print(f"\nModul: {modul_name}")

        modul_information = []
        
        # Aktivitäten im Modul durchlaufen
        for activity in module.findall('activity'):
            # Informationen zu den Aktivitäten auslesen
            wochentag = activity.get('weekdays')
            startzeit = activity.get('starttime')
            endzeit = activity.get('endtime')
            location = activity.get('locations')
            typ = activity.get('type')
            typ_lesbar = mapping.get(typ, typ)
            desc = activity.get('desc')
            

            # Ausgabe der Aktivitätsinformationen
            if wochentag and location:
                modul_information.append({"wochentag": wochentag, 
                                          "startzeit": startzeit, 
                                          "endzeit": endzeit, 
                                          "location": location, 
                                          "typ": typ_lesbar, 
                                          "description": desc})
                print(f"{typ_lesbar} in Raum {location} am {wochentag}: {startzeit} - {endzeit}", end="")
                if desc:
                    print(f" ({desc})", end="")
                print("")
        inf[modul_name] = modul_information
            
else:
    print(f"Fehler bei der Anfrage: Statuscode {response.status_code}")

print(inf)


Modul: Einführung in die Programmierung
Praktikum in Raum Is-H409 am Fr: 08:00:00 - 09:30:00
Praktikum in Raum Is-H409 am Do: 14:00:00 - 15:30:00
Übung in Raum Is-C301 am Fr: 14:00:00 - 15:30:00
Vorlesung in Raum Is-C301 am Fr: 12:00:00 - 13:30:00

Modul: Grundlagen der Informatik 1
Klausur in Raum Is-Audimax am Mo: 14:00:00 - 15:30:00
Klausur in Raum Is-Audimax am Mo: 14:00:00 - 15:30:00
Seminar in Raum Is-H206 am Do: 09:00:00 - 11:30:00
Seminar in Raum Is-H206 am Mi: 12:00:00 - 14:30:00
Seminar in Raum Is-H206 am Mi: 13:00:00 - 15:30:00
Vorlesung in Raum Is-P301 am Mo: 14:00:00 - 15:30:00 (ab Nov M301)

Modul: Mathematik 1
Vorlesung in Raum Is-H205 am Mi: 08:00:00 - 08:15:00
Übung in Raum Is-H213 am Fr: 12:00:00 - 13:30:00
Übung in Raum Is-H402/403 am Fr: 10:00:00 - 11:30:00 (Ersatztermin 03.10.)
Übung in Raum Is-H402/403 am Fr: 08:00:00 - 09:30:00
Übung in Raum Is-H401 am Di: 08:00:00 - 09:30:00
Übung in Raum Is-H411 am Di: 12:00:00 - 13:30:00
Vorlesung in Raum Is-Audimax am Mi: 08

Mit dem folgenden Code werden alle Veranstalungen ausgelesen. Über die URL stehen die Informationen zu allen verwendeten Räumen an einem Tag und einem Standort zur Verfügung. Durch die Verwendung der 4 Standorte und der letzen 14 Tage lasse sich so alle Veranstaltungen mit allen Terminen auslesen.

In [ ]:
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

today = datetime.today()

last_14_days = [today - timedelta(days=i) for i in range(14)]

formatted_dates = [date.strftime('%Y-%m-%d') for date in last_14_days]

# Dicts für die Informationen
vpis_location = {}
vpis_name = {}
vpis_room = {}
vpis_employee = {}

location_strings = ["Iserlohn", "Hagen", "Meschede", "Soest"]

# URL für die GET Anfrage
#url = "https://vpis.fh-swf.de/WS2025/raum.php3?Raum=&Standort=Is&Template=XML"
#url = "http://vpis.fh-swf.de/aktuell.php3/Raum?Template=XML&Standort=Is"
#url = "https://vpis.fh-swf.de/WS2025/raum.php3?Raum=&Standort=Is&Template=XML&Tag=2025-10-04"

#https://vpis.fh-swf.de/WS2025/raum.php3?Raum=#SPLUSA8968A&Standort=Is&Template=XML

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)"
                  " Chrome/117.0.0.0 Safari/537.36"
}

for loc in location_strings:

    base_url = f'https://vpis.fh-swf.de/WS2025/raum.php3?Raum=&Standort={loc[:2]}&Template=XML&Tag='
    
    urls = [base_url + date for date in formatted_dates]

    for url in urls:

        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            # XML-Inhalt auslesen
            xml_content = response.content
            
            # XML parsen und bei falschem XML Content mit nächster URL fortsetzen
            try:
                root = ET.fromstring(xml_content)
            except Exception:
                continue

            staffs = root.find('staffs')

            employees = {}
            """
            STAFF TAG:

            <staff surname="Mustermann" forename="Max" title1="Dr." title2="Prof.">
            <name>Mustermann, M.</name>
            </staff>
            """
            for staff in staffs.findall('staff'):
                if 'surname' in staff.attrib:
                    parts = [
                        (staff.get('title2') or '').strip(),
                        (staff.get('title1') or '').strip(),
                        staff.get('forename') or '',
                        staff.get('surname') or ''
                    ]
                    key = staff.findtext('name')
                    value = ' '.join(part for part in parts if part)

                    employees[key] = value


            locations_description = {}

            locations = root.find('locations')

            """
            LOCATION TAG:

            <location size="378" sizeklausur="101">
            <name>RAUMNAME</name>
            <hostkey semesterjahr="WS2025">#SPLUS38CC3B</hostkey>
            <description>RAUMBESCHREIBUNG</description>
            <location-suitabilities>
                <location-suitability semesterjahr="WS2025" primary="N" secondary="J">#SPLUS0AD361</location-suitability>
                <location-suitability semesterjahr="WS2025" primary="J" secondary="N">#SPLUS0B2757</location-suitability>
                <location-suitability semesterjahr="WS2025" primary="J" secondary="N">#SPLUS20DC50</location-suitability>
                <location-suitability semesterjahr="WS2025" primary="J" secondary="N">#SPLUS5F1FD1</location-suitability>
                <location-suitability semesterjahr="WS2025" primary="J" secondary="N">#SPLUS8000E4</location-suitability>
                <location-suitability semesterjahr="WS2025" primary="J" secondary="N">#SPLUS9E087E</location-suitability>
                <location-suitability semesterjahr="WS2025" primary="J" secondary="N">#SPLUSE55B14</location-suitability>
            </location-suitabilities>
            </location>
            """
            for location in locations.findall('location'):
                key = location.findtext('name')
                value = location.findtext('description')
                locations_description[key] = value



            activities = root.find('activities')
            # Aktivitäten auslesen
            for activity in activities.findall('activity'):

                act = {}

                name = activity.findtext('name')
                act['name'] = name

                activity_type = activity.findtext('activity-type')
                act['activity_type'] = activity_type

                termine = []

                for date_elem in activity.findall('./activity-dates/activity-date'):
                    termine.append({'date': date_elem.get('date'), 'begin':  date_elem.get('begin'), 'end': date_elem.get('end')})

                act['dates'] = termine

                ort = activity.findtext('./activity-locations/activity-location')
                act['room'] = ort
            
                lehrpersonal = [p.text for p in activity.findall('./activity-staffs/activity-staff')]

                # kurzen Namen aus der Aktivität mit dem langen Namen aus dem staff Tag ersetzen
                names = [employees[key] for key in lehrpersonal if key in employees]
                act['employees'] = names

                # Informationen in den dicts nach unterschiedlichen Schlüsseln speichern
                if loc not in vpis_location:
                    vpis_location[loc] = []
                if act not in vpis_location[loc]:
                    vpis_location[loc].append(act)

                if name not in vpis_name:
                    vpis_name[name] = []
                if act not in vpis_name[name]:
                    vpis_name[name].append(act)

                if ort not in vpis_room:
                    vpis_room[ort] = []
                if act not in vpis_room[ort]:
                    vpis_room[ort].append(act)

                for employee in names:
                    if employee not in vpis_employee:
                        vpis_employee[employee] = []
                    if act not in vpis_employee[employee]:
                        vpis_employee[employee].append(act)
                

            
                    
        else:
            print(f"Fehler bei der Anfrage: Statuscode {response.status_code}")



Daten speichern

In [ ]:
import pickle
import os

current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..', '..'))

vpis_data_path = os.path.join(project_root, 'data', 'vpis')


with open(os.path.join(vpis_data_path, "vpis_location.pkl"), "wb") as f:
    pickle.dump(vpis_location, f)
with open(os.path.join(vpis_data_path, "vpis_employee.pkl"), "wb") as f:
    pickle.dump(vpis_employee, f)
with open(os.path.join(vpis_data_path, "vpis_name.pkl"), "wb") as f:
    pickle.dump(vpis_name, f)
with open(os.path.join(vpis_data_path, "vpis_room.pkl"), "wb") as f:
    pickle.dump(vpis_room, f)

In [15]:
for v in vpis_room["Is-H401"]:
    print(v)

{'name': 'Mathematik 1', 'activity_type': 'Übung', 'dates': [{'date': '2025-09-30', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-10-07', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-10-14', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-10-28', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-11-04', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-11-11', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-11-18', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-11-25', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-12-02', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-12-09', 'begin': '08:00', 'end': '09:30'}, {'date': '2025-12-16', 'begin': '08:00', 'end': '09:30'}, {'date': '2026-01-06', 'begin': '08:00', 'end': '09:30'}, {'date': '2026-01-13', 'begin': '08:00', 'end': '09:30'}, {'date': '2026-01-20', 'begin': '08:00', 'end': '09:30'}], 'room': 'Is-H401', 'employees': ['Dipl.-Ing. Ulrike Hinze']}
{'name': 'Fertigungsverfahren Grundlagen - Metallzerspanung', 'ac

Informationen aus einer ICS Datei auslesen

In [ ]:
from ics import Calendar

# Link für ICS Dateien
# https://vpis.fh-swf.de/WS2025/student.php3/auth/icalendar.ics?Template=ICS

#username = "benutzername"
#password = "passwort"

#response = requests.get(url, auth=(username, password))

#if response.status_code == 200:
#    ics_content = response.text

with open(os.path.join(vpis_data_path, 'icalendar.ics'), 'r', encoding='utf-8') as file:
    kalender = Calendar(file.read())

for event in kalender.events:
    #print(event)
    if event.categories != {'Feiertag'}:
        print(event.name, event.begin, event.end, event.location)


Netzökonomie / Praktikum 2025-09-27T12:45:00+02:00 2025-09-27T14:15:00+02:00 Is-P203
Netzökonomie / Praktikum 2025-12-06T12:45:00+01:00 2025-12-06T14:15:00+01:00 Is-P203
Netzökonomie / Praktikum 2025-11-08T12:45:00+01:00 2025-11-08T14:15:00+01:00 Is-P203
Netzökonomie / Praktikum 2025-10-11T12:45:00+02:00 2025-10-11T14:15:00+02:00 Is-P203
Netzökonomie / Praktikum 2025-09-13T12:45:00+02:00 2025-09-13T14:15:00+02:00 Is-P203
Netzökonomie / Praktikum 2025-12-20T12:45:00+01:00 2025-12-20T14:15:00+01:00 Is-P203
Netzökonomie / Praktikum 2025-11-22T12:45:00+01:00 2025-11-22T14:15:00+01:00 Is-P203
